# GeoSat Full-City Inference (Google Colab)

Run one complete city at a time on a Colab GPU. The pipeline downloads Sentinel-2 imagery for two years, runs ResNet-50 land-cover inference, creates the change map and area statistics, then copies the finished city folder to Google Drive.

## Before running

1. Select a GPU runtime in **Runtime > Change runtime type**.
2. Upload `models/resnet50_best.pt` and `service_account.json` to the Drive folder configured below.
3. Set `CITY` to one city name from `config/cities.json`.
4. Run the cells from top to bottom.

The default city is `Mumbai`, which uses the full Mumbai bounding box. To test the setup with a smaller download first, temporarily use `TinyMumbai`.

Outputs are saved to `LandCoverChangeResults/<CITY>/` in Google Drive.

In [ ]:
# 1. Mount Google Drive and verify the Colab GPU
from google.colab import drive
drive.mount('/content/drive')

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Choose Runtime > Change runtime type > T4 GPU (or another GPU) and rerun."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Configure one full-city run
# Change only CITY when processing another city.
CITY = "Mumbai"
START_YEAR = 2019
END_YEAR = 2023
BATCH_SIZE = 32
FORCE_RERUN = False

DRIVE_ROOT = "/content/drive/MyDrive/LandCoverChangeResults"
DRIVE_MODEL = f"{DRIVE_ROOT}/models/resnet50_best.pt"
DRIVE_GEE_CREDENTIALS = f"{DRIVE_ROOT}/service_account.json"

REPO_URL = "https://github.com/saran318/Geo_Sat.git"
REPO_DIR = "/content/Geo_Sat"
PROJECT_DIR = f"{REPO_DIR}/geo_sat project"

if START_YEAR == END_YEAR:
    raise ValueError("START_YEAR and END_YEAR must be different.")

print(f"Configured city: {CITY}")
print(f"Comparison: {START_YEAR} -> {END_YEAR}")

In [ ]:
# 3. Install dependencies, clone the project, and prepare Drive credentials
import os
import shutil
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "earthengine-api",
        "rasterio",
        "pandas",
        "Pillow",
        "python-dotenv",
        "pyproj",
    ],
    check=True,
)

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(PROJECT_DIR)
os.makedirs("models", exist_ok=True)

if not os.path.isfile(DRIVE_MODEL):
    raise FileNotFoundError(f"Model not found in Google Drive: {DRIVE_MODEL}")
if not os.path.isfile(DRIVE_GEE_CREDENTIALS):
    raise FileNotFoundError(f"GEE service account not found in Google Drive: {DRIVE_GEE_CREDENTIALS}")

shutil.copy2(DRIVE_MODEL, "models/resnet50_best.pt")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = DRIVE_GEE_CREDENTIALS

print(f"Project ready: {PROJECT_DIR}")
print("Model copied from Drive and GEE credentials configured.")

In [ ]:
# 4. Validate the selected city and prepare resumable status tracking
import datetime
import json

with open("config/cities.json", "r", encoding="utf-8") as file:
    cities = json.load(file)

if CITY not in cities:
    raise ValueError(f"Unknown city '{CITY}'. Available cities: {', '.join(cities)}")

city_drive_dir = os.path.join(DRIVE_ROOT, CITY)
completed_file = os.path.join(city_drive_dir, "completed.json")
status_file = os.path.join(DRIVE_ROOT, "processing_status.json")
os.makedirs(DRIVE_ROOT, exist_ok=True)

if not FORCE_RERUN and os.path.isfile(completed_file):
    print(f"{CITY} is already completed in Drive: {city_drive_dir}")
    print("Set FORCE_RERUN = True to process it again.")
else:
    bbox = cities[CITY]["bbox"]
    print(f"Validated {CITY}; bounding box = {bbox}")
    print(f"Results will be copied to {city_drive_dir}")

In [ ]:
# 5. Run the complete city pipeline and sync its results to Drive
import gc
import time

if not FORCE_RERUN and os.path.isfile(completed_file):
    print(f"Skipping {CITY}; completed.json already exists.")
else:
    os.makedirs(city_drive_dir, exist_ok=True)
    if os.path.isfile(status_file):
        with open(status_file, "r", encoding="utf-8") as file:
            processing_status = json.load(file)
    else:
        processing_status = {}

    processing_status[CITY] = {
        "status": "running",
        "start_time": datetime.datetime.now().isoformat(),
    }
    with open(status_file, "w", encoding="utf-8") as file:
        json.dump(processing_status, file, indent=2)

    start_time = time.time()
    try:
        command = [
            sys.executable,
            "run_real_pipeline.py",
            "--city",
            CITY,
            "--years",
            str(START_YEAR),
            str(END_YEAR),
            "--batch_size",
            str(BATCH_SIZE),
        ]
        subprocess.run(command, check=True)

        local_results = os.path.join("data", "results", CITY)
        if not os.path.isdir(local_results):
            raise FileNotFoundError(f"Pipeline output folder not found: {local_results}")

        for item in os.listdir(local_results):
            source = os.path.join(local_results, item)
            destination = os.path.join(city_drive_dir, item)
            if os.path.isdir(source):
                if os.path.exists(destination):
                    shutil.rmtree(destination)
                shutil.copytree(source, destination)
            else:
                shutil.copy2(source, destination)

        duration = round(time.time() - start_time, 2)
        processing_status[CITY] = {
            "status": "completed",
            "start_time": processing_status[CITY]["start_time"],
            "end_time": datetime.datetime.now().isoformat(),
            "duration_sec": duration,
            "output_path": city_drive_dir,
        }
        print(f"Completed {CITY} in {duration} seconds.")
        print(f"Drive output: {city_drive_dir}")

    except Exception as error:
        processing_status[CITY].update({
            "status": "failed",
            "end_time": datetime.datetime.now().isoformat(),
            "error": str(error),
        })
        raise
    finally:
        with open(status_file, "w", encoding="utf-8") as file:
            json.dump(processing_status, file, indent=2)
        torch.cuda.empty_cache()
        gc.collect()

In [ ]:
# 6. Confirm the expected outputs
earlier_year, later_year = sorted((START_YEAR, END_YEAR))
expected_outputs = [
    os.path.join(city_drive_dir, "completed.json"),
    os.path.join(city_drive_dir, "metadata.json"),
    os.path.join(city_drive_dir, "predictions", f"classified_{earlier_year}.tif"),
    os.path.join(city_drive_dir, "predictions", f"classified_{later_year}.tif"),
    os.path.join(city_drive_dir, "change_maps", f"change_{earlier_year}_{later_year}.tif"),
    os.path.join(city_drive_dir, "statistics", "area_stats.csv"),
]

missing_outputs = [path for path in expected_outputs if not os.path.exists(path)]
if missing_outputs:
    raise FileNotFoundError("Missing expected outputs:\n" + "\n".join(missing_outputs))

print(f"Verified {len(expected_outputs)} outputs for {CITY}.")
print(f"All files are available in: {city_drive_dir}")